# 🐄 DeVaca — Ahorro Sugerido con IA
## DevIAthon 2025 · Categoría Deuna · Fintech

**Problema:** Solo el 48% de usuarios activos de Deuna genera ingresos directos para la plataforma.

**Solución:** Sistema de ahorro sugerido inteligente que convierte usuarios pasivos en activos recurrentes mediante IA aplicada a datos de comportamiento financiero.

---

**Colores corporativos usados en todo el notebook**
- 🟢 Verde Deuna: `#00DDA6`
- 🟣 Morado Deuna: `#432959`

**Equipo:** Deuna_team1 · UPEC — Tulcán, Ecuador


## 1. 📊 Contexto y Problema

En la actualidad solo el **48% de los usuarios activos de Deuna** genera ingresos directos para la plataforma. El otro 52% representa una oportunidad de monetización desaprovechada.

### Los 4 ejes de monetización de Deuna

1. **Compras en comercios** afiliados (MiPymes y grandes superficies).
2. **Pagos de servicios** básicos (luz, agua, internet).
3. **Recargas móviles** de operadoras.
4. **Transferencias interbancarias** salientes.

### Los 4 perfiles de usuario identificados

| Perfil | Descripción |
|---|---|
| 🛌 **El Dormido** | Tiene la app instalada pero no la activa. |
| 🤨 **El Escéptico** | Desconfía del pago digital, opera montos pequeños. |
| 👻 **El Invisible** | Usa Deuna desde el portal de su banco sin saberlo. |
| 🎁 **El Cazabonos** | Se registró por el bono de bienvenida y desapareció. |

DeVaca propone una capa de **ahorro sugerido gamificado** que, apoyada en IA, convierte cada transacción cotidiana en una oportunidad de ahorro y de ingreso recurrente para Deuna.


In [ ]:
# Librerias estandar de Google Colab (sin instalaciones adicionales)
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler

from itertools import permutations

# Paleta corporativa Deuna
VERDE = "#00DDA6"
MORADO = "#432959"
VERDE_SOFT = "#7FECCF"   # variante clara del verde
MORADO_SOFT = "#8E73A6"  # variante clara del morado
GRIS_FONDO = "#F9FAFB"
ROJO_META = "#EF4444"

PALETA_PERFILES = {
    "El Dormido": MORADO,
    "El Escéptico": MORADO_SOFT,
    "El Invisible": VERDE_SOFT,
    "El Cazabonos": VERDE,
}

# Estilo limpio y profesional para todas las graficas
sns.set_theme(style="white", context="notebook")
plt.rcParams.update({
    "axes.spines.top": False,
    "axes.spines.right": False,
    "axes.edgecolor": "#9CA3AF",
    "axes.labelcolor": "#374151",
    "xtick.color": "#6B7280",
    "ytick.color": "#6B7280",
    "axes.titleweight": "bold",
    "axes.titlecolor": MORADO,
    "axes.titlesize": 13,
    "axes.labelsize": 11,
    "font.family": "DejaVu Sans",
    "figure.facecolor": "white",
    "axes.facecolor": "white",
})

print("Entorno listo · paleta Deuna cargada")


## 2. 🗃️ Generación de Datos Simulados

Generamos un dataset de **500 usuarios** simulados que refleja los comportamientos observados en la base real de Deuna. Las variables capturan los tres ejes que la IA usará después para segmentar y predecir.


In [ ]:
# Semilla para reproducibilidad de toda la simulacion
np.random.seed(42)

N = 500

# Generamos cada variable con distribuciones que imitan el comportamiento real
df = pd.DataFrame({
    "usuario_id": np.arange(1, N + 1),

    # Frecuencia de uso semanal — sesgo a la baja (muchos usuarios pasivos)
    "frecuencia_semanal": np.clip(
        np.random.exponential(scale=1.8, size=N), 0.1, 7.0
    ).round(2),

    # Monto promedio por transaccion — distribucion log-normal (tickets pequenos predominan)
    "monto_promedio_transaccion": np.clip(
        np.random.lognormal(mean=2.6, sigma=0.7, size=N), 1.0, 150.0
    ).round(2),

    # Cuantos de los 4 ejes de monetizacion usa el usuario (1 a 4)
    "tipos_pago_usados": np.random.choice([1, 2, 3, 4], size=N, p=[0.45, 0.30, 0.18, 0.07]),

    # Dias desde la ultima transaccion (1 a 90)
    "dias_desde_ultima_transaccion": np.random.randint(1, 91, size=N),

    # Usa la app Deuna directo (True) o desde el portal de su banco (False)
    "usa_app_directo": np.random.choice([True, False], size=N, p=[0.55, 0.45]),

    # Recibio el bono de registro
    "recibio_bono_registro": np.random.choice([True, False], size=N, p=[0.38, 0.62]),

    # Antiguedad del usuario en dias (1 semana a 2 anos)
    "antiguedad_dias": np.random.randint(7, 731, size=N),

    # Etiqueta objetivo: 48% generan ingresos para Deuna
    "genera_ingresos": np.random.choice([True, False], size=N, p=[0.48, 0.52]),
})

print(f"Dataset generado: {len(df)} usuarios · {df.shape[1]} variables\n")
display(df.head())
print("\nEstadisticas descriptivas:")
display(df.describe().round(2))


## 3. 🤖 Modelo 1 — Segmentación de Usuarios con K-Means

Aplicamos **K-Means** con `k=4` sobre las variables de comportamiento (frecuencia, monto, diversidad de pagos e inactividad) para descubrir los 4 perfiles esperados. Luego mapeamos automáticamente cada cluster al perfil de negocio correspondiente analizando los centroides.


In [ ]:
# 1) Seleccionamos las variables de comportamiento
features = [
    "frecuencia_semanal",
    "monto_promedio_transaccion",
    "tipos_pago_usados",
    "dias_desde_ultima_transaccion",
]
X = df[features].values

# 2) Normalizamos: las variables tienen escalas muy distintas
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# 3) Entrenamos K-Means con k=4
kmeans = KMeans(n_clusters=4, random_state=42, n_init=10)
df["cluster"] = kmeans.fit_predict(X_scaled)

# 4) Estadisticas por cluster (incluyendo las variables binarias para el mapeo)
stats_cols = features + ["usa_app_directo", "recibio_bono_registro", "antiguedad_dias"]
cluster_stats = df.groupby("cluster")[stats_cols].mean()

# 5) Normalizamos las estadisticas entre clusters para construir un score por perfil
norm = (cluster_stats - cluster_stats.min()) / (
    cluster_stats.max() - cluster_stats.min() + 1e-9
)

def score_perfil(cid, perfil):
    s = norm.loc[cid]
    if perfil == "El Dormido":
        # Muchos dias inactivo y baja frecuencia
        return s["dias_desde_ultima_transaccion"] - s["frecuencia_semanal"]
    if perfil == "El Escéptico":
        # Monto bajo y pocos tipos de pago
        return -s["monto_promedio_transaccion"] - s["tipos_pago_usados"]
    if perfil == "El Invisible":
        # Usa la app directa muy poco (entra desde su banco)
        return -s["usa_app_directo"]
    if perfil == "El Cazabonos":
        # Recibio bono y poca antiguedad
        return s["recibio_bono_registro"] - s["antiguedad_dias"]
    return 0.0

# 6) Probamos las 24 permutaciones posibles y elegimos la asignacion que maximiza el score total
perfiles = ["El Dormido", "El Escéptico", "El Invisible", "El Cazabonos"]
clusters = list(cluster_stats.index)

mejor_perm = max(
    permutations(perfiles),
    key=lambda perm: sum(score_perfil(c, p) for c, p in zip(clusters, perm)),
)
mapa_cluster_perfil = dict(zip(clusters, mejor_perm))
df["perfil"] = df["cluster"].map(mapa_cluster_perfil)

# 7) Tabla resumen con las caracteristicas promedio de cada perfil
resumen = df.groupby("perfil")[stats_cols].mean().round(2)
resumen["usuarios"] = df["perfil"].value_counts()
resumen = resumen.loc[perfiles]  # orden estable

print("Mapeo cluster → perfil:")
for c, p in mapa_cluster_perfil.items():
    print(f"  cluster {c} → {p}")
print()
display(resumen)


In [ ]:
# Visualizacion en 2 subplots lado a lado
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

# ── Subplot 1: scatter frecuencia vs dias inactivo ──
ax1 = axes[0]
for perfil in perfiles:
    sub = df[df["perfil"] == perfil]
    ax1.scatter(
        sub["frecuencia_semanal"],
        sub["dias_desde_ultima_transaccion"],
        c=PALETA_PERFILES[perfil],
        label=perfil,
        alpha=0.75,
        s=40,
        edgecolors="white",
        linewidths=0.5,
    )

ax1.set_xlabel("Frecuencia semanal (veces / semana)")
ax1.set_ylabel("Días desde última transacción")
ax1.set_title("Segmentación de usuarios por comportamiento")
ax1.legend(frameon=False, loc="upper right", fontsize=9)
ax1.grid(axis="y", linestyle="--", alpha=0.3)

# ── Subplot 2: bar chart horizontal con distribucion de perfiles ──
ax2 = axes[1]
conteo = df["perfil"].value_counts().reindex(perfiles)
ax2.barh(conteo.index, conteo.values, color=VERDE, edgecolor="white", linewidth=1.2)

for i, v in enumerate(conteo.values):
    ax2.text(v + 4, i, f"{v}", va="center", fontsize=10, color="#374151")

ax2.set_xlabel("Cantidad de usuarios")
ax2.set_title("Distribución de perfiles")
ax2.invert_yaxis()
ax2.grid(axis="x", linestyle="--", alpha=0.3)

plt.tight_layout()
plt.show()


## 4. 📈 Modelo 2 — Predicción de Meta de Ahorro

Para cada perfil de usuario activo proyectamos cuándo alcanzará la meta inicial de **$30** usando una regresión lineal simple (`np.polyfit` grado 1) sobre 7 semanas de historial. La proyección permite mostrar al usuario en la app: *"A este ritmo llegarás a tu meta el día X del mes"*.


In [ ]:
# Historial semanal de ahorro (en USD) de 3 usuarios representativos
usuario_activo   = [2.5, 5.8, 9.2, 14.1, 19.3, 24.8, 28.5]
usuario_moderado = [1.0, 2.5, 4.0,  6.2,  8.9, 11.3, 14.0]
usuario_nuevo    = [0.5, 1.2, 2.8,  4.1,  5.5,  7.2,  9.0]

usuarios = {
    "Usuario Activo": usuario_activo,
    "Usuario Moderado": usuario_moderado,
    "Usuario Nuevo": usuario_nuevo,
}

META = 30.0
semanas = np.arange(1, 8)

filas = []
for nombre, data in usuarios.items():
    # Regresion lineal de grado 1: y = slope * semana + intercept
    slope, intercept = np.polyfit(semanas, data, 1)

    # Despeje: semana donde y = 30
    semana_meta = (META - intercept) / slope
    dia_mes_meta = int(np.ceil(semana_meta * 7))
    dia_actual = int(semanas[-1] * 7)
    dias_restantes = max(0, dia_mes_meta - dia_actual)

    filas.append({
        "Perfil": nombre,
        "Ahorro actual": f"${data[-1]:.2f}",
        "Ritmo semanal": f"${slope:.2f}/sem",
        "Día estimado para meta": dia_mes_meta,
        "Días restantes": dias_restantes,
    })

predicciones = pd.DataFrame(filas)
display(predicciones)

# Guardamos para usar en la siguiente grafica
predicciones_raw = {nombre: np.polyfit(semanas, data, 1) for nombre, data in usuarios.items()}


In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))

colores_usuarios = {
    "Usuario Activo": VERDE,
    "Usuario Moderado": MORADO,
    "Usuario Nuevo": MORADO_SOFT,
}

semanas_proyectadas = np.arange(1, 9)  # extendemos 1 semana mas

for nombre, data in usuarios.items():
    color = colores_usuarios[nombre]
    slope, intercept = predicciones_raw[nombre]

    # Linea solida: datos reales (semanas 1-7)
    ax.plot(semanas, data, marker="o", color=color, linewidth=2.4,
            label=f"{nombre} (real)", markersize=7, markeredgecolor="white",
            markeredgewidth=1.2)

    # Linea punteada: proyeccion (semana 7-8 y mas alla si hace falta)
    y_proy = slope * semanas_proyectadas + intercept
    ax.plot(semanas_proyectadas, y_proy, linestyle="--", color=color,
            linewidth=1.6, alpha=0.7,
            label=f"{nombre} (proyección)")

# Linea de meta
ax.axhline(META, color=ROJO_META, linestyle=":", linewidth=1.8, alpha=0.8)
ax.text(8.05, META + 0.6, "Meta: $30", color=ROJO_META, fontsize=10, fontweight="bold")

ax.set_xlabel("Semana")
ax.set_ylabel("Ahorro acumulado (USD)")
ax.set_title("Proyección de ahorro por perfil de usuario")
ax.set_xticks(semanas_proyectadas)
ax.legend(frameon=False, loc="upper left", fontsize=9, ncol=2)
ax.grid(axis="y", linestyle="--", alpha=0.3)
ax.set_ylim(0, max(META + 5, max(usuario_activo) + 5))

plt.tight_layout()
plt.show()


## 5. 💡 Modelo 3 — Ahorro Sugerido Personalizado por IA

El monto sugerido **no es genérico**: la IA lo calcula a partir de la frecuencia y el ticket promedio de cada usuario. Cuanto más activo es el usuario, mayor porcentaje del ticket sugerimos ahorrar. Así el sistema empuja al alza a quienes ya están comprometidos sin asustar a los que apenas empiezan.


In [ ]:
def calcular_ahorro_sugerido(usuario):
    """Calcula el monto optimo a sugerir como ahorro segun el perfil de comportamiento.

    Regla:
      - Usuario muy activo (>= 4/sem) con ticket alto (>= $20): 12% del ticket
      - Usuario activo (>= 2/sem) con ticket medio (>= $10): 10% del ticket
      - Usuario poco frecuente (< 2/sem): 8% del ticket
    """
    frec = usuario["frecuencia_semanal"]
    monto = usuario["monto_promedio_transaccion"]

    if frec >= 4 and monto >= 20:
        sugerido = monto * 0.12
    elif frec >= 2 and monto >= 10:
        sugerido = monto * 0.10
    else:
        sugerido = monto * 0.08

    return round(sugerido, 2)

# Aplicamos la funcion a los 500 usuarios
df["ahorro_sugerido"] = df.apply(calcular_ahorro_sugerido, axis=1)

# Estadisticas por perfil
stats_ahorro = df.groupby("perfil")["ahorro_sugerido"].agg(["mean", "median", "min", "max"]).round(2)
stats_ahorro.columns = ["Promedio", "Mediana", "Mínimo", "Máximo"]
stats_ahorro = stats_ahorro.loc[perfiles]
print("Ahorro sugerido promedio por perfil (USD):")
display(stats_ahorro)

# Boxplot del ahorro sugerido por perfil
fig, ax = plt.subplots(figsize=(10, 5))
data_box = [df[df["perfil"] == p]["ahorro_sugerido"].values for p in perfiles]
bplot = ax.boxplot(
    data_box,
    labels=perfiles,
    patch_artist=True,
    medianprops=dict(color=MORADO, linewidth=2),
    whiskerprops=dict(color="#9CA3AF"),
    capprops=dict(color="#9CA3AF"),
    flierprops=dict(marker="o", markerfacecolor=VERDE_SOFT,
                    markeredgecolor="white", markersize=5, alpha=0.7),
)
for patch in bplot["boxes"]:
    patch.set_facecolor(VERDE)
    patch.set_alpha(0.55)
    patch.set_edgecolor(VERDE)

ax.set_ylabel("Ahorro sugerido (USD)")
ax.set_title("Distribución del ahorro sugerido por perfil de usuario")
ax.grid(axis="y", linestyle="--", alpha=0.3)

plt.tight_layout()
plt.show()


In [ ]:
# Proyectamos el impacto de DeVaca sobre la base real de Deuna
USUARIOS_TOTALES = 10_000
COMISION = 0.015  # 1.5% por transaccion

# Distribucion de perfiles segun los clusters encontrados
distribucion = df["perfil"].value_counts(normalize=True).reindex(perfiles)

# Tasa de activacion estimada al introducir DeVaca (cuanto del segmento se vuelve generador de ingresos)
tasa_activacion = {
    "El Dormido":   0.35,
    "El Escéptico": 0.25,
    "El Invisible": 0.40,
    "El Cazabonos": 0.30,
}

filas_impacto = []
for perfil in perfiles:
    n_segmento = int(USUARIOS_TOTALES * distribucion[perfil])
    n_activados = int(n_segmento * tasa_activacion[perfil])

    sub = df[df["perfil"] == perfil]
    frec_promedio = sub["frecuencia_semanal"].mean()
    monto_promedio = sub["monto_promedio_transaccion"].mean()

    tx_mensuales = n_activados * frec_promedio * 4  # 4 semanas
    ingreso_mensual = tx_mensuales * monto_promedio * COMISION

    filas_impacto.append({
        "Perfil": perfil,
        "Usuarios segmento": n_segmento,
        "Activados con DeVaca": n_activados,
        "Tx adicionales / mes": int(round(tx_mensuales)),
        "Ingreso adicional / mes (USD)": round(ingreso_mensual, 2),
    })

impacto = pd.DataFrame(filas_impacto)

# Fila de totales
total_row = {
    "Perfil": "TOTAL",
    "Usuarios segmento": impacto["Usuarios segmento"].sum(),
    "Activados con DeVaca": impacto["Activados con DeVaca"].sum(),
    "Tx adicionales / mes": impacto["Tx adicionales / mes"].sum(),
    "Ingreso adicional / mes (USD)": round(impacto["Ingreso adicional / mes (USD)"].sum(), 2),
}
impacto_total = pd.concat([impacto, pd.DataFrame([total_row])], ignore_index=True)

print("Impacto proyectado de DeVaca sobre 10,000 usuarios de Deuna\n")
display(impacto_total)

# Grafico de barras del ingreso adicional por perfil
fig, ax = plt.subplots(figsize=(10, 5))
ax.bar(
    impacto["Perfil"],
    impacto["Ingreso adicional / mes (USD)"],
    color=MORADO,
    edgecolor="white",
    linewidth=1.5,
)
for i, v in enumerate(impacto["Ingreso adicional / mes (USD)"]):
    ax.text(i, v + (impacto["Ingreso adicional / mes (USD)"].max() * 0.02),
            f"${v:,.0f}", ha="center", fontsize=10, fontweight="bold", color=MORADO)

ax.set_ylabel("Ingreso adicional mensual (USD)")
ax.set_title("Impacto proyectado de DeVaca en monetización Deuna")
ax.grid(axis="y", linestyle="--", alpha=0.3)

plt.tight_layout()
plt.show()

# Guardamos el incremento porcentual para usarlo en las conclusiones
USUARIOS_GENERAN_HOY = int(USUARIOS_TOTALES * 0.48)
NUEVOS_GENERADORES = impacto["Activados con DeVaca"].sum()
INCREMENTO_PCT = NUEVOS_GENERADORES / USUARIOS_GENERAN_HOY * 100

print(f"\nUsuarios que generan ingresos HOY: {USUARIOS_GENERAN_HOY:,}")
print(f"Nuevos generadores con DeVaca:     +{NUEVOS_GENERADORES:,}")
print(f"Incremento proyectado:             +{INCREMENTO_PCT:.1f}%")
print(f"Ingreso adicional mensual total:   ${total_row['Ingreso adicional / mes (USD)']:,.2f}")


## 6. ✅ Conclusiones

- **DeVaca convierte usuarios pasivos en activos** mediante un mecánismo de ahorro gamificado anclado a transacciones reales. El microahorro es indoloro porque el usuario ya iba a gastar.
- **La IA segmenta correctamente los 4 perfiles** con K-Means sobre 4 variables de comportamiento y mapea cada cluster a su perfil de negocio sin etiquetas manuales.
- **El ahorro sugerido es personalizado**, no genérico: el porcentaje aplicado al ticket varía según frecuencia y monto, evitando fricción en usuarios nuevos y maximizando captura en activos.
- **La proyección muestra que el incremento de usuarios generadores de ingresos es relevante** — la celda anterior calcula el `INCREMENTO_PCT` exacto sobre el 48% inicial.
- **Responde a los 4 ejes de monetización de Deuna** porque cada transacción —comercios, servicios, recargas o transferencias— puede activar la sugerencia de ahorro.
- **Escalable a toda la base de usuarios** sin infraestructura nueva: solo requiere consumir el historial transaccional ya existente.


## 🐄 DeVaca · DevIAthon 2025

**Equipo:** Deuna_team1
**Universidad:** UPEC — Tulcán, Ecuador
**Categoría:** Fintech · IA aplicada a datos
